In [1]:
import os
import sys
import configparser

#add dark to path
src_code_dir = os.sep.join(os.path.abspath('').split(os.sep)[:-2])+os.sep
sys.path.insert(0,src_code_dir)

from dark.gateway import DarkGateway
from dark import DarkMap
from dark import DarkPid
from dark.decoder import DarkDecoder

# deployer
from deployer import DarkDeployer

#LOG SETUP 
import logging
logging.basicConfig(level=logging.INFO)

load blockchain drivers, parameters and configurations

In [2]:
PROJECT_ROOT = 'D:\\workspace\\dark\\dARK\\'
PROJECT_ROOT = '/home/thiago/workspace/dark/dARK'

config_file = os.path.join(PROJECT_ROOT,'config.ini')
#blockchain config
bc_config = configparser.ConfigParser()
bc_config.read(config_file)
#deployed contracts config
deployed_contracts_config = configparser.ConfigParser()
deployed_contracts_config.read(os.path.join(PROJECT_ROOT,'deployed_contracts.ini'))

# Load Blockchain Drivers to the dARK GateWay
dgw =  DarkGateway(bc_config,deployed_contracts_config)
# create a map to interact with the Blockchain Smart Contracts
dm = DarkMap(dgw)


# Deploy & COnfigure

## Deploy

In [ ]:
def populate_file_list(dir,files):
    """
        Private method to populate a list with full path of the smartcontrats
    """
    lista = []
    for i in files:
        lista.append( os.path.join(dir,i) )
    return lista

def save_smart_contract(deployed_contracts_dict,compiled_contracts_dict,deployed_contracts_config_path):
    """
        Save the deployed smart contracts
        - inputs the deploed_contracts dict

        - It is essential to configure the contract prior to its usage
        - Please only use this method to save configured contracts
    """
    config = configparser.ConfigParser()

    already_config_set = set()
    for deployed_sc_name in deployed_contracts_dict.keys():
        for compiled_sc_name in compiled_contracts_dict.keys():

            if deployed_sc_name.split('.')[0] == compiled_sc_name.split(':')[-1]:
            # if (dsc_name.endswith(csc_name.split('.')[0])) and (dsc_name not in already_config_set):
                addr = deployed_contracts_dict[deployed_sc_name]
                abi = compiled_contracts_dict[compiled_sc_name]['abi']
                # print(deployed_sc_name,addr,compiled_sc_name,abi)
                config[deployed_sc_name] = {'addr' : addr, 'abi' : abi}

    with open(deployed_contracts_config_path, 'w') as configfile:
        config.write(configfile)


In [ ]:
config = configparser.ConfigParser()
config.read(config_file)

#PATH SETUP
DAPP_ROOT = os.path.join(PROJECT_ROOT, config['base']['dapp_dir'])
LIB_PATH = os.path.join(DAPP_ROOT, config['smartcontracts']['lib_dir'])
UTIL_PATH = os.path.join(DAPP_ROOT, config['smartcontracts']['util_dir'])
DB_PATH = os.path.join(DAPP_ROOT, config['smartcontracts']['db_dir'])
SERVICE_PATH = os.path.join(DAPP_ROOT, config['smartcontracts']['service_dir'])

#CONTRACTS 
libs_path = populate_file_list(LIB_PATH,config['smartcontracts']['lib_files'].split())
utils_path = populate_file_list(UTIL_PATH,config['smartcontracts']['utils_files'].split())
dbs_paths = populate_file_list(DB_PATH,config['smartcontracts']['dbs_files'].split())
services_path = populate_file_list(SERVICE_PATH,config['smartcontracts']['service_files'].split())

In [ ]:
dark_deployer = DarkDeployer(dgw)
compiled_contracts = dark_deployer.compile_all(services_path + dbs_paths + utils_path + libs_path)
c = dark_deployer.deploy_contracts(compiled_contracts)
save_smart_contract(c,compiled_contracts,'./deployed_contracts.ini')

#### Debug Deploy

In [ ]:
k = '/home/thiago/workspace/dark/dARK/dARK_dapp/services/PIDService.sol:PIDService'
cc = compiled_contracts[k]

a = dgw.deploy_contract_besu(cc)

In [ ]:
addr = a

code = dgw.w3.eth.get_code(addr)
print(f"Código do contrato ({addr}):", code.hex())
print("Tamanho:", len(code), "bytes")

In [ ]:
def get_contract_address(name):
    """
        Get the address of the contract
    """
    for k in compiled_contracts.keys():
        if name in k:
            return compiled_contracts[k]['address']
    return None

def check_contract_address(addr):

    # for k in compiled_contracts.keys():
    #     if name in k:
    #         addr = compiled_contracts[k]['address']
    #         if addr != '0x
    code = dgw.w3.eth.get_code(addr)
    
    print(dgw.w3.eth.get_storage_at(addr,0))
    print(code)
    print("Código do contrato:", code.hex())

check_contract_address(a)


In [ ]:
check_contract_address(a)
check_contract_address('0x9e699d6c7ccf183F0B09675A9E867d1486EEF85b')

In [ ]:
# sc.constructor().estimate_gas()
est_gas = sc.constructor().estimate_gas({
    'from': dgw.authority_addr
})

In [ ]:
dgw.authority_addr

In [ ]:
tx_receipt

In [ ]:
addr = tx_receipt.contractAddress

code = dgw.w3.eth.get_code(addr)
print(f"Código do contrato ({addr}):", code.hex())
print("Tamanho:", len(code), "bytes")

In [ ]:
k = '/home/thiago/workspace/dark/dARK/dARK_dapp/services/PIDService.sol:PIDService'
cc = compiled_contracts[k]
bytecode = cc['bin']
if not bytecode.startswith('0x'):
    bytecode = '0x' + bytecode

sc = dgw.w3.eth.contract( abi=cc['abi'],
                                bytecode=bytecode
                                )

tx_param = dgw.get_tx_params(2000000)
# tx_param['gasPrice'] = dgw.w3.to_wei('20', 'gwei')


tx = sc.constructor().build_transaction(tx_param)
# print(tx)

signed_tx = dgw.w3.eth.account.sign_transaction(tx, private_key='0xae6ae8e5ccbfb04590405997ee2d52d2b330726137b875053c36d94e974d162f')
print(signed_tx)
tx_hash = dgw.w3.eth.send_raw_transaction(signed_tx.rawTransaction)
print(tx_hash.hex())

tx_receipt = dgw.w3.eth.wait_for_transaction_receipt(tx_hash.hex())
print("Contrato implantado em:", tx_receipt.contractAddress)


# dgw.w3.is_connected(), dgw.w3.eth.accounts
# tx_hash = sc.constructor().transact()
# dgw.deploy_contract_besu(compiled_contracts[k])

# for k in compiled_contracts.keys():
#     print('Deploying contract: %s' % k)
#     print(dgw.deploy_contract_besu(compiled_contracts[k]))


In [ ]:
# Criar contrato
# contract_interface = compiled_contracts[k]
# bytecode = contract_interface['bin']
# if not bytecode.startswith('0x'):
#     bytecode = '0x' + bytecode

# contract = dgw.w3.eth.contract( abi=contract_interface['abi'],
#                         bytecode=bytecode
#                         )

# tx_param = dgw.get_tx_params(2000000)

# # 'gasPrice': dgw.w3.to_wei('20', 'gwei')

# tx = contract.constructor().build_transaction(tx_param)


# gas_estimate = dgw.w3.eth.estimate_gas(tx)
# print(f"Gas estimate: {gas_estimate}")


# # Obter nonce
# nonce = w3.eth.get_transaction_count(deployer_address)

# # Construir transação de deploy
# transaction = contract.constructor().build_transaction({
#     'from': deployer_address,
#     'nonce': nonce,
#     'gas': 2000000,  # limite de gás
#     'gasPrice': w3.toWei('50', 'gwei'),  # preço do gás
# })

# # Assinar a transação
# signed_txn = w3.eth.account.sign_transaction(transaction, private_key=private_key)

# # Enviar a transação
# tx_hash = w3.eth.send_raw_transaction(signed_txn.rawTransaction)

# # Esperar confirmação
# tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

# # Endereço do contrato implantado
# contract_address = tx_receipt.contractAddress
# print(f"Contrato implantado em: {contract_address}")


In [ ]:
contract_interface = compiled_contracts[k]

bytecode = contract_interface['bin']
if not bytecode.startswith('0x'):
    bytecode = '0x' + bytecode

sc = dgw.w3.eth.contract( abi=contract_interface['abi'],
                        bytecode=bytecode
                        )

transaction = sc.constructor().prepare_transaction({
    'from': dgw.authority_addr,
    # outros parâmetros
})

# transaction = sc.constructor().buildTransaction({
#     'from': dgw.authority_addr,
#     # opcionalmente, 'nonce', 'gasPrice' etc.
# })

gas_estimate = dgw.w3.eth.estimate_gas(transaction)

# sc.constructor().estimate_gas()

In [ ]:
# bytecode = contract_interface['bin']
# if not bytecode.startswith('0x'):
#     bytecode = '0x' + bytecode

# # Opcional: verificar se todos os caracteres após '0x' são hexadecimais
# hex_part = bytecode[2:]
# if not all(c in '0123456789abcdefABCDEF' for c in hex_part):
#     raise ValueError("Bytecode contém caracteres não hexadecimais.")

## Configure

In [ ]:
import logging
deployed_contracts_config_path = os.path.join(PROJECT_ROOT,'deployed_contracts.ini')
noid_provider_config_path = os.path.join(PROJECT_ROOT,'noid_provider_config.ini')

# LOAD CONFIGURATION
config = configparser.ConfigParser()
config.read(config_file)

#LOG SETUP 
logging.basicConfig(level=logging.INFO)

In [ ]:
dark_deployer = DarkDeployer(dgw)
# dark_deployer.setup_dark_onchain_contracts(deployed_contracts_config_path)
# dark_deployer.configure_noid_provider(deployed_contracts_config_path,noid_provider_config_path)

In [ ]:

dark_deployer = DarkDeployer(dgw)

#configure smart contracts
dark_deployer.setup_dark_onchain_contracts(deployed_contracts_config_path)
#configure payloadschema
# dark_deployer.configure_payload_schema(deployed_contracts_config_path,config_file)
#configure noid_provider
dark_deployer.configure_noid_provider(deployed_contracts_config_path,noid_provider_config_path)

### Configure Debug

In [ ]:

# auth_db_addr = smart_contract_config['AuthoritiesDB.sol']['addr']
# contract_addr = smart_contract_config['AuthoritiesService.sol']['addr']
# contract_interface = smart_contract_config['AuthoritiesService.sol']['abi']
# # print(contract_addr,contract_interface)
# auth_service = dwg.w3.eth.contract(address=contract_addr, abi=ast.literal_eval(contract_interface))

# dm.auth_service.address
# 0x4AC87179077491910B90b49a5120aF5aFb3CFFEc
# 0x24F137C6340E28B8f24770400081ac4b4a66BDe7

In [ ]:
print(dm.auth_service.address)
code = dgw.w3.eth.get_code(dm.auth_service.address)
print("Código do contrato:", code.hex())

dm.auth_service.functions.get_db().call()


In [ ]:
def decode_logs(dw:DarkGateway, receipt, contract_abi):
    # Cria um contrato temporário apenas para decodificação
    contract = dw.w3.eth.contract(abi=contract_abi)

    decoded_logs = []

    for log in receipt['logs']:
        try:
            # Decodifica o log usando a ABI
            decoded_event = contract.events.log_id().processLog(log)
            decoded_logs.append(decoded_event)
        except Exception as e:
            # Se não for o evento esperado, ignora ou trata
            print(f"Erro ao decodificar log: {e}")
            continue

    return decoded_logs

In [ ]:
a = {'blockHash': '0x76a47a6b10ae5f7903377930b905be49f2c5d8397a3d878be23fee54fef7d72c', 'blockNumber': 3418740, 'contractAddress': None, 'cumulativeGasUsed': 24048, 'from': '0xf17f52151EbEF6C7334FAD080c5704D77216b732', 'gasUsed': 24048, 'effectiveGasPrice': 100000000000, 'logs': [], 'logsBloom': '0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'status': 1, 'to': '0x6027f80a8F4eB1dbe1B77B7dECDe5Cc9da6d5Ec3', 'transactionHash': '0xe06f428cc8e8b06461b24aa987fecd9832f093deb4fa4adaedb6d580b162f435', 'transactionIndex': 0, 'type': 0}


# Exemplo de uso após obter a receipt
# '0xe06f428cc8e8b06461b24aa987fecd9832f093deb4fa4adaedb6d580b162f435'
logs_decodificados = decode_logs(dgw,a, dm.auth_service.abi)

for event in logs_decodificados:
    print(f"Evento log_id emitido: {event['args']}")


# Dark Blockchain Method Invocation

## Sync Example

requesting a pid

In [4]:
a_pid = dm.sync_request_pid()
a_pid

'8033/fkwf3000000028'

retrieving a pid

In [ ]:
pid_python_object = dm.get_pid_by_ark(a_pid)
pid_python_object.to_dict()

In [ ]:
try:
    dm.sync_add_external_pid(pid_python_object.pid_hash,'10.1016/j.is.2021.101826')
    pid_python_object = dm.get_pid_by_ark(a_pid)
    pid_python_object.to_dict()
except ValueError:
    print("NEED TO ADD A URL FIRST")

In [ ]:
dm.sync_set_url(pid_python_object.pid_hash,'https://www.sciencedirect.com/science/article/abs/pii/S0306437921000661')
pid_python_object = dm.get_pid_by_ark(a_pid)
pid_python_object.to_dict()

In [ ]:
dm.sync_add_external_pid(pid_python_object.pid_hash,'10.1016/j.is.2021.101826')
pid_python_object = dm.get_pid_by_ark(a_pid)
pid_python_object.to_dict()

In [ ]:
payload_data = { 'title' : 'Blockchain-based Privacy-Preserving Record Linkage: enhancing data privacy in an untrusted environment.' ,
                 'author' : 'Thiago Nóbrega',
                #  'type' : 'Article'
            }

tx_set3 = dm.sync_set_payload(pid_python_object.pid_hash,payload_data)

In [ ]:
pid_python_object = dm.get_pid_by_ark(a_pid)
pid_python_object.to_dict()

## Assync Example

pid request

In [ ]:
a_pid = dm.sync_request_pid()
a_pid

In [ ]:
pid_python_object = dm.get_pid_by_ark(a_pid)
pid_python_object.to_dict()

set

In [ ]:
#retrieve the pid_hash
pid_hash = pid_python_object.pid_hash
pid_hash

In [ ]:
#retive a ARK id from a pid_hash
ark_to_search = dm.convert_pid_hash_to_ark(pid_hash)
#notice that is the same of the original pid
print(a_pid,ark_to_search)

this transaction will fail because the dARK still a draft (because it do not have a url)...

In [ ]:
#now lets addd a url and remove the draft status from the pid
tx_set2 = dm.async_set_url(pid_hash,'https://www.sciencedirect.com/science/article/abs/pii/S0169023X2300040X')


In [ ]:
tx_status, tx_recipt = dgw.transaction_was_executed(tx_set2)
tx_status

In [ ]:
tx_set1 = dm.async_set_external_pid(pid_hash,'10.1016/j.datak.2023.102180')
# tx_status, tx_recipt = dgw.transaction_was_executed(tx_set1)
# tx_status

In [ ]:
tx_status, tx_recipt = dgw.transaction_was_executed(tx_set1)
tx_status

In [ ]:
pid_python_object_2 = dm.get_pid_by_ark(a_pid).to_dict()
# notice that there is a new PID
pid_python_object_2

In [ ]:
pid_python_object_1 = dm.get_pid_by_ark(ark_to_search).to_dict()
# notice that there is a new PID
pid_python_object_1

In [ ]:
payload_data = { 'title' : 'Towards automatic Privacy-Preserving Record Linkage: A Transfer Learning based classification step' ,
                 'author' : 'Thiago Nóbrega',
                #  'type' : 'Article'
            }

tx_set3 = dm.async_set_payload(pid_hash,payload_data)

In [ ]:
tx_set3

In [ ]:
for tx in tx_set3:
    tx_status, tx_recipt = dgw.transaction_was_executed(tx)
    print(tx.hex(), tx_status)

In [ ]:
pid_python_object_3 = dm.get_pid_by_ark(a_pid).to_dict()
# notice that there is a new PID
pid_python_object_3

# Log example

decoding 

In [ ]:
#create a dARK Transaction Decoder
dc = DarkDecoder(dgw)

In [ ]:
dc.extract_dark_data(tx_set1)

dc.extract_dark_data(tx_set2)

# Multiple User Example

In [ ]:
PROJECT_ROOT = 'D:\\workspace\\dark\\dARK\\'
config_file = os.path.join(PROJECT_ROOT,'config.ini')
#blockchain config
bc_config = configparser.ConfigParser()
bc_config.read(config_file)
#deployed contracts config
deployed_contracts_config = configparser.ConfigParser()
deployed_contracts_config.read(os.path.join(PROJECT_ROOT,'deployed_contracts.ini'))

passing user key as argument to the DarkGateway

In [ ]:
user_pk = '0xae6ae8e5ccbfb04590405997ee2d52d2b330726137b875053c36d94e974d162f'
# Load Blockchain Drivers to the dARK GateWay
dgw =  DarkGateway(bc_config,deployed_contracts_config,account_private_key=user_pk)
# create a map to interact with the Blockchain Smart Contracts
dm = DarkMap(dgw)